# CLAUDIO: wake word training

Trains the two wake word models, `hey_claudio.onnx` and `claudio.onnx`, from synthetic voices
and your own recordings. Free; it takes 4-6 hours and the CPU runtime is enough. Each model is trained with three
settings and the one that hears your held-out takes best is kept.

*Addestra i due modelli della wake word con voci sintetiche e le tue registrazioni. Gratis,
4-6 ore; basta il runtime CPU, la GPU non serve. Ogni modello è addestrato con tre
impostazioni e tiene la migliore sulle tue registrazioni di prova.*

1. **Runtime → Run all** (*Runtime → Esegui tutto*).
2. When asked, upload `claudio_training.zip`, made on the PC by
   `uv run python scripts/record_samples.py pack`
   (*quando te lo chiede, carica lo zip creato sul PC*).
3. Keep this tab open until the end: the browser then downloads `claudio_wakeword.zip`
   (*tieni aperta la scheda: alla fine il browser scarica lo zip con i modelli*).

If the session disconnects, press **Run all** again: finished steps, and the settings already
trained, are skipped as long as the session's disk is still there (*se si disconnette, premi
di nuovo Esegui tutto: i passi e le impostazioni già finiti si saltano*). If a
step fails, Run all stops there: copy the last red lines and send them to Claude (*se un
passo fallisce Esegui tutto si ferma lì: copia le ultime righe rosse e mandale a Claude*).

In [ ]:
# 1. Check the machine / Controllo della macchina
import os
import shutil
import sys

free_gb = shutil.disk_usage("/content").free / 2**30
print(f"Python {sys.version.split()[0]}, {os.cpu_count()} CPU, {free_gb:.0f} GB liberi")
assert free_gb > 30, "Servono almeno 30 GB liberi sul disco della sessione"

In [ ]:
# 2. Install the pinned tools / Installazione (versioni fissate)
%pip install -q piper-tts==1.8.0 onnxruntime==1.31.0 onnx==1.23.2
%pip install -q --no-deps openwakeword==0.6.0
import importlib.util

import torch  # already in Colab, like scipy, soundfile and scikit-learn

missing = [m for m in ("scipy", "soundfile", "sklearn") if importlib.util.find_spec(m) is None]
assert not missing, f"Mancano {missing}: aggiungili con %pip install"
print("torch", torch.__version__, "| GPU" if torch.cuda.is_available() else "| CPU")

In [ ]:
# @title 3. Upload claudio_training.zip / Carica lo zip
import io
import json
import pathlib
import zipfile

from google.colab import files

# new_recordings: upload a new zip into a session that already has older recordings
# (carica uno zip nuovo in una sessione che ne ha già uno).
new_recordings = False  # @param {type:"boolean"}
# force: train again even models that are finished (rifà anche i modelli già pronti).
force = False  # @param {type:"boolean"}
os.chdir("/content")
if new_recordings or not pathlib.Path("claudio/wakeword/train_wakeword.py").is_file():
    uploaded = files.upload()
    assert uploaded, "Nessun file caricato: premi di nuovo Esegui tutto e scegli lo zip"
    # From memory: a second upload of the same name is saved as "... (1).zip".
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as zf:
        names = zf.namelist()
        found = {
            kind: sum(n.startswith(f"samples/{kind}/") and n.endswith(".wav") for n in names)
            for kind in ("hey_claudio", "claudio")
        }
        assert "wakeword/train_wakeword.py" in names and all(found.values()), (
            "Questo non è lo zip di registrazioni (claudio_training.zip, fatto con "
            f"record_samples.py pack): dentro ci sono {found}. Premi di nuovo Esegui tutto"
        )
        shutil.rmtree("claudio_new", ignore_errors=True)
        zf.extractall("claudio_new")  # nothing of the old session is touched until it worked
    shutil.rmtree("claudio/samples", ignore_errors=True)
    shutil.copytree("claudio_new", "claudio", dirs_exist_ok=True)
    shutil.rmtree("claudio_new")
os.chdir("/content/claudio")
for kind in ("hey_claudio", "claudio", "negatives", "noise"):
    print(kind, len(list(pathlib.Path("samples", kind).glob("*.wav"))), "registrazioni")
# The scripts travel inside the zip: an old zip trains the old way.
!python wakeword/train_wakeword.py version
force_flag = "--force" if force else ""

In [ ]:
# 4. Download training data, voices and noise (about 20 GB) / Scarica dati, voci e rumori
!python wakeword/train_wakeword.py download --data data
# A failed ! command does not stop Run all on its own: _exit_code is what it returned.
if _exit_code:  # noqa: F821
    raise RuntimeError("Download non riuscito: leggi le righe NO qui sopra")

In [ ]:
# 5. Train "Hey Claudio" / Addestra "Hey Claudio"
!python wakeword/train_wakeword.py train --model hey_claudio --workers {os.cpu_count()} {force_flag}
if _exit_code:  # noqa: F821
    raise RuntimeError('Addestramento di "Hey Claudio" non riuscito: leggi le righe qui sopra')

In [ ]:
# 6. Train "Claudio" / Addestra "Claudio"
!python wakeword/train_wakeword.py train --model claudio --workers {os.cpu_count()} {force_flag}
if _exit_code:  # noqa: F821
    raise RuntimeError('Addestramento di "Claudio" non riuscito: leggi le righe qui sopra')

In [ ]:
# 7. Download the models / Scarica i modelli
wanted = [f"{m}{end}" for m in ("hey_claudio", "claudio") for end in (".onnx", "_report.json")]
missing = [name for name in wanted if not pathlib.Path("output", name).is_file()]
assert not missing, f"Mancano {missing}: guarda gli errori delle celle 5 e 6"
for m in ("hey_claudio", "claudio"):
    done = json.loads(pathlib.Path("output", f"{m}_report.json").read_text())
    for problem in done.get("problems", []):
        print(f"ATTENZIONE {m}: il modello {problem}")
    if not done.get("meets_target", True):
        print(f"ATTENZIONE {m}: nessuna soglia arriva al limite di falsi allarmi")
shutil.make_archive("/content/claudio_wakeword", "zip", "output")
files.download("/content/claudio_wakeword.zip")

## Next / Dopo

Unzip `claudio_wakeword.zip` and copy `hey_claudio.onnx` and `claudio.onnx` into
`models/wakeword/` in the CLAUDIO folder. Send Claude the two `*_report.json` files (or a
screenshot of the tables above): they hold the recommended thresholds.

*Estrai lo zip, copia i due `.onnx` in `models/wakeword/` e manda a Claude i due
`*_report.json` (o una foto delle tabelle qui sopra): contengono le soglie consigliate.*